# 00. Environment Setup and BigQuery Initialization

## 0. Overview
This notebook initializes the analysis environment and verifies connectivity to the MIMIC-IV BigQuery datasets. It serves as the entry point for the original cohort/association workflow (01–04b) and the separate prediction-time evaluation (04c).

No analytic transformations are performed in this notebook.

## 1. Purpose
The purpose of this notebook is to ensure a reproducible and consistent execution environment, including Python dependencies, virtual environment configuration, and BigQuery authentication.

## 2. Data Sources
- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`
- Working dataset: `nonicu_raas`

## 3. Environment Setup
- Python virtual environment activation
- Package version checks
- Jupyter kernel verification

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

**Live input behavior:** Normal execution requires the configured real BigQuery inputs and approved access. Missing settings, credentials, permissions or tables stop execution with a cause and a next step; no synthetic/demo data or saved results are substituted. Stored presentation-replay outputs are review artifacts, not a successful new data load. Use a fresh kernel for a new run.


In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
# Confirm the exact Python interpreter path.
import sys

print(sys.executable)


In [ ]:
# Core imports and project configuration.
from google.api_core.exceptions import NotFound
from google.auth import default
from google.cloud import bigquery

PROJECT_ID = "mimic-iv-portfolio"
DATASET_ID = "nonicu_raas"
LOCATION = "US"

# Lowercase aliases are kept for consistency with downstream notebooks.
project_id = PROJECT_ID
dataset_id = DATASET_ID


In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 00.1'])



## 4. BigQuery Initialization
- Authentication and client initialization
- Verification of BigQuery access
- Minimal test query against MIMIC-IV tables

In [ ]:
from real_data_access import real_data_access, real_data_io
# Authenticate with Application Default Credentials (ADC) and initialize BigQuery once.
# Run `gcloud auth application-default login` outside this notebook if ADC is not configured.
with real_data_access(project_id, dataset_id):
    credentials, adc_project_id = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    client = bigquery.Client(credentials=credentials, project=project_id)

print("Credentials type:", type(credentials).__name__)
print("ADC project:", adc_project_id)
print("BigQuery client project:", client.project)


def dataset_reference(dataset_name: str = dataset_id) -> bigquery.Dataset:
    """Return a BigQuery Dataset object for the configured project and dataset."""
    dataset = bigquery.Dataset(f"{project_id}.{dataset_name}")
    dataset.location = LOCATION
    return dataset


@real_data_io
def ensure_dataset_exists(dataset_name: str = dataset_id) -> bigquery.Dataset:
    """Create the working dataset if it does not already exist."""
    dataset = dataset_reference(dataset_name)
    try:
        return client.get_dataset(dataset)
    except NotFound:
        return client.create_dataset(dataset)


In [ ]:
# Ensure the working dataset exists in the configured location.
dataset = ensure_dataset_exists(dataset_id)
print(f"Dataset `{dataset.full_dataset_id}` is available in {dataset.location}.")

## 5. Sanity Checks
- Confirmation of successful BigQuery connection
- Validation of dataset and table visibility

In [ ]:
with notebook_results.capture('Table 00.1', globals()):
    query = """
SELECT COUNT(*) AS icu_stays
FROM `physionet-data.mimiciv_3_1_icu.icustays` # Extract Base ICU Stays (mimiciv_icu.icustays)
"""
    df = client.query(query).to_dataframe()
    _notebook_result_value = (print(df))


## 6. Outputs and Downstream Use
This notebook produces no analytic dataset. 
 
These checks establish environment and connectivity prerequisites; they do not by themselves validate every downstream analysis.

The public workflow covers **BigQuery and Python notebooks 00–04c**. Notebooks 01–04b describe the original retrospective association branch; [04c](04c_prediction_time_evaluation.ipynb) separately defines admission/24-hour ward-confirmed prediction cohorts.

Select the repository-local Python 3.12 `.venv` kernel. Before using a different BigQuery destination, follow the coordinated SQL/notebook/helper replacements in [REPRODUCIBILITY](../REPRODUCIBILITY.md#replace-the-author-specific-bigquery-destination-before-running); the project labels above are examples, not automatic configuration. Notebook 01–02 construction cells use `CREATE OR REPLACE`.

Saved real-data execution evidence is in the [current execution record](../docs/CLINICAL_REEXECUTION_615de01.md). Local executed outputs are retained; Git strips outputs and execution counts from public notebooks. This explanatory update does not represent a new execution.
